In [ ]:
#| default_exp icons

In [ ]:
#| hide
import sys
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory
from PIL import Image

One square image to the icon assets each platform's bundle wants.

Pillow does the drawing, and `kavacha[icons]` installs it. Every function imports it at the call rather than at module import. On a machine without Pillow, importing `kavacha.icons` works, and the `ImportError` comes from the call that makes an icon.

A source image is refused unless it is square and at least 512px across. macOS draws an app icon at 1024.

The `.ico` and the favicon are Pillow's work and are written on any platform. An `.icns` is `iconutil`'s work, and only macOS has `iconutil`.

In [ ]:
#| export
from __future__ import annotations
import subprocess
from fastcore.all import Path

In [ ]:
#| export
#: The sizes an `.icns` carries, as `(pixels, iconset name)`. Retina entries are the same pixel
#: count as the next size up, which is why `icon_16x16@2x` and `icon_32x32` are both 32.
ICNS_SIZES = ((16, 'icon_16x16'), (32, 'icon_16x16@2x'), (32, 'icon_32x32'),
              (64, 'icon_32x32@2x'), (128, 'icon_128x128'), (256, 'icon_128x128@2x'),
              (256, 'icon_256x256'), (512, 'icon_256x256@2x'), (512, 'icon_512x512'),
              (1024, 'icon_512x512@2x'))

#: What a Windows `.ico` holds. Above 256 the format stores a PNG, and nothing asks for one.
ICO_SIZES = (16, 24, 32, 48, 64, 128, 256)

`ICNS_SIZES` is a pixel count and the file name `iconutil` reads it under. A retina entry carries the same pixel count as the plain entry one size up, so `icon_16x16@2x` and `icon_32x32` are both 32px, and both files are written. Ten files over seven distinct pixel counts.

`ICO_SIZES` stops at 256. Above that the `.ico` format stores a PNG rather than a bitmap, and nothing Windows draws asks for one.

In [ ]:
ICNS_SIZES[:4]

((16, 'icon_16x16'),
 (32, 'icon_16x16@2x'),
 (32, 'icon_32x32'),
 (64, 'icon_32x32@2x'))

In [ ]:
#| hide
test_eq(len({n for _, n in ICNS_SIZES}), len(ICNS_SIZES))          # one file each, no name written twice
test_eq(sorted({s for s, _ in ICNS_SIZES}), [16, 32, 64, 128, 256, 512, 1024])
test_eq([s for s, n in ICNS_SIZES if n == 'icon_16x16@2x'], [32])
test_eq(max(ICO_SIZES), 256)

In [ ]:
#| export
def _open(src):
    "The source image as RGBA, square, or the reason it cannot be used."
    from PIL import Image
    im = Image.open(src).convert('RGBA')
    if im.width != im.height:
        raise ValueError(f'{src} is {im.width}x{im.height}; an app icon has to be square')
    if im.width < 512:
        raise ValueError(f'{src} is {im.width}px; macOS draws icons at 1024 and will upscale this')
    return im

def _resized(im, size):
    from PIL import Image
    return im.resize((size, size), Image.LANCZOS)

`_open` is the gate every function here goes through. It returns the source as RGBA, so a JPEG that has no alpha channel gains one, and a mark drawn on a plate the OS supplies keeps its transparency.

It raises `ValueError` on an image that is not square, and on one under 512px across. Neither is repaired here. Only the source image can fix a mark that is stretched, or too small for macOS to draw at 1024.

`_resized` is Lanczos, Pillow's slowest and best filter. An icon is resized once, at build time.

In [ ]:
#| hide
#| exec_doc
tmp = TemporaryDirectory(); root = Path(tmp.name)
def mkpng(p, size=1024, colour=(24, 28, 44, 255)):
    Image.new('RGBA', (size, size), colour).save(p); return Path(p)
logo = mkpng(root/'logo.png')
mkpng(root/'small.png', size=128)
Image.new('RGB', (1024, 1024), (198, 40, 40)).save(root/'logo.jpg')

In [ ]:
_open(root/'logo.jpg').mode, _open(root/'logo.jpg').size

('RGBA', (1024, 1024))

The two refusals, in the words a caller sees. The temp folder these examples work in stands in as `/proj`.

In [ ]:
Image.new('RGBA', (800, 400)).save(root/'wide.png')
for p in ('wide.png', 'small.png'):
    try: _open(root/p)
    except ValueError as e: print(str(e).replace(str(root), '/proj'))

/proj/wide.png is 800x400; an app icon has to be square
/proj/small.png is 128px; macOS draws icons at 1024 and will upscale this


In [ ]:
#| export
def iconset(src, out):
    "The `.iconset` folder `iconutil` turns into an `.icns`. Returns the folder."
    im, out = _open(src), Path(out)
    out.mkdir(parents=True, exist_ok=True)
    for size, name in ICNS_SIZES: _resized(im, size).save(out/f'{name}.png')
    return out

def icns(src, out, keep_iconset=False):
    """An `.icns` from one square image, through macOS's own `iconutil`.

    `iconutil` is macOS-only, so this raises everywhere else rather than writing something a
    bundle would accept and then draw wrongly.
    """
    out = Path(out)
    folder = out.with_suffix('.iconset')
    iconset(src, folder)
    try: subprocess.run(['iconutil', '-c', 'icns', str(folder), '-o', str(out)], check=True,
                        capture_output=True)
    except FileNotFoundError as e:
        raise RuntimeError('an .icns is built by `iconutil`, which only macOS has') from e
    finally:
        if not keep_iconset:
            for f in folder.glob('*.png'): f.unlink()
            folder.rmdir()
    return out

`iconset` writes the ten PNGs into a folder it creates, and returns the folder. An existing folder is written into rather than emptied first.

`icns` puts that folder next to `out`, runs `iconutil` over it, and removes it again. `keep_iconset` leaves it in place. The removal happens whether `iconutil` ran or not, so a call that failed leaves no folder of PNGs behind.

Off macOS there is no `iconutil` and `icns` raises `RuntimeError` naming it. No `.icns` is written.

In [ ]:
out = iconset(logo, root/'Set.iconset')
sorted((Image.open(p).size[0], p.stem) for p in out.glob('*.png'))

[(16, 'icon_16x16'),
 (32, 'icon_16x16@2x'),
 (32, 'icon_32x32'),
 (64, 'icon_32x32@2x'),
 (128, 'icon_128x128'),
 (256, 'icon_128x128@2x'),
 (256, 'icon_256x256'),
 (512, 'icon_256x256@2x'),
 (512, 'icon_512x512'),
 (1024, 'icon_512x512@2x')]

In [ ]:
try: print(icns(logo, root/'Demo.icns').name)
except RuntimeError as e: print(e)

an .icns is built by `iconutil`, which only macOS has


In [ ]:
#| hide
assert not (root/'Demo.iconset').exists(), 'the folder goes whether iconutil ran or not'

In [ ]:
#| export
def ico(src, out, sizes=ICO_SIZES):
    "A Windows `.ico` from one square image. Pillow writes every size into the one file."
    im, out = _open(src), Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    im.save(out, format='ICO', sizes=[(s, s) for s in sizes])
    return out

`ico` writes every size into the one file. Windows takes the size it is drawing out of it, so an app ships one icon file rather than seven. The parent directory is created.

`sizes` narrows the set. Pillow silently drops an entry above 256, and an entry larger than the source. A size the format cannot hold is missing from the file rather than reported.

In [ ]:
with Image.open(ico(logo, root/'assets'/'Demo.ico')) as im: print(sorted(im.info['sizes']))
with Image.open(ico(logo, root/'small.ico', sizes=(16, 32, 512))) as im: print(sorted(im.info['sizes']))

[(16, 16), (24, 24), (32, 32), (48, 48), (64, 64), (128, 128), (256, 256)]
[(16, 16), (32, 32)]


In [ ]:
#| hide
test_eq(Image.open(root/'small.ico').info['sizes'], {(16, 16), (32, 32)})   # 512 asked for, 512 dropped

In [ ]:
#| export
def favicon(src, out, size=64):
    """A `.png` favicon for the app's own web UI, so the tab and the Dock carry one mark.

    Small: below about 64px a detailed mark stops carrying its line work, and a browser tab is
    16px of it.
    """
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    _resized(_open(src), size).save(out)
    return out

def icons_for(src, assets, name):
    "Every icon asset a bundle wants, written under `assets`. Skips the ones this platform cannot make."
    import sys
    assets, out = Path(assets), {}
    assets.mkdir(parents=True, exist_ok=True)
    out['favicon'] = str(favicon(src, assets/'favicon.png'))
    out['ico'] = str(ico(src, assets/f'{name}.ico'))
    if sys.platform == 'darwin': out['icns'] = str(icns(src, assets/f'{name}.icns'))
    return out

`favicon` is one PNG for the app's own web UI, so a tab and the Dock carry the same mark. 64px by default. A tab draws 16px of it, and a detailed mark stops carrying its line work below about 64.

`icons_for` writes what a bundle wants under `assets`, and returns each path as a `str` keyed by the kind. `favicon` and `ico` are written everywhere. `icns` is written on macOS only, and elsewhere its key is absent. Every path in the dict names a file that is there.

In [ ]:
made = icons_for(logo, root/'assets', 'Demo')
{k: Path(v).name for k, v in made.items()}

{'favicon': 'favicon.png', 'ico': 'Demo.ico'}

In [ ]:
#| hide
test_eq(set(made) - {'icns'}, {'favicon', 'ico'})
test_eq('icns' in made, sys.platform == 'darwin')
assert all(Path(p).exists() for p in made.values()), 'every path returned is a file that is there'
test_eq(Image.open(made['favicon']).size, (64, 64))
test_eq(Image.open(favicon(logo, root/'web'/'favicon.png', size=32)).size, (32, 32))

In [ ]:
#| hide
tmp.cleanup()